# Notebook de Mise à Jour des Valeurs Réelles

Ce notebook permet de mettre à jour les prédictions de consommation électrique avec les valeurs réelles de la veille.

## Fonctionnalités :
1. Configuration de la base de données
2. Récupération des prédictions de la veille
3. Génération de valeurs aléatoires (pour l'instant)
4. Mise à jour des enregistrements dans la base de données
5. Vérification des mises à jour

## Structure de la table de stockage :
- `prediction_id`: TEXT (UUID)
- `prediction_timestamp`: TIMESTAMP
- `prediction_date`: TIMESTAMP
- `prediction_index`: INTEGER
- `prediction`: DOUBLE PRECISION
- `confidence`: DOUBLE PRECISION
- `model_version`: TEXT
- `actual_value`: DOUBLE PRECISION (nouveau champ)
- `created_at`: TIMESTAMP

## 0. Initialisation et Configuration

In [1]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


## 1. Configuration de la Base de Données

In [2]:
# Configuration PostgreSQL (à adapter selon votre environnement)
DB_URI = os.getenv("PREDICTIONS_POSTGRES_URI")

if DB_URI:
    print("📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI")
else:
    DB_CONFIG = {
        "host": os.getenv("DB_HOST", "localhost"),
        "port": os.getenv("DB_PORT", "5432"),
        "database": os.getenv("DB_NAME", "jinsudai"),
        "user": os.getenv("DB_USER", "postgres"),
        "password": os.getenv("DB_PASSWORD", ""),
    }
    print(f"📊 Configuration DB :")
    print(f"  Host: {DB_CONFIG['host']}")
    print(f"  Port: {DB_CONFIG['port']}")
    print(f"  Database: {DB_CONFIG['database']}")
    
    # Construire l'URI PostgreSQL
    DB_URI = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    print(f"  URI construite")

📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI


## 2. Initialisation du Pipeline de Valeurs Réelles

In [3]:
import sys
sys.path.insert(0, str(Path.cwd() / "src"))

from ml.utils.pipelines.Actual_values_pipeline import ActualValuesPipeline

# Initialiser le pipeline
pipeline = ActualValuesPipeline(db_uri=DB_URI)

print("🔧 Pipeline de valeurs réelles initialisé")
print(f"  URI DB: {'Configurée' if DB_URI else 'Non configurée'}")

c:\Users\SustCoop\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\ml-6PV5fMfH-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-12 04:46:33,696 - ml.utils.pipelines.Actual_values_pipeline - INFO - Pipeline de valeurs réelles initialisé


🔧 Pipeline de valeurs réelles initialisé
  URI DB: Configurée


## 3. Configuration du Pipeline

In [4]:
print("🔧 Configuration du pipeline...")

try:
    if not pipeline.setup():
        raise RuntimeError("Échec de la configuration du pipeline")
    
    print("✅ Pipeline configuré avec succès")
    print("  - Connexion à la base de données vérifiée")
    print("  - Colonne actual_value ajoutée si nécessaire")
    
except Exception as e:
    logger.error(f"Erreur lors de la configuration: {e}")
    raise

2026-06-12 04:46:33,734 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 1: CONFIGURATION ===


🔧 Configuration du pipeline...


2026-06-12 04:46:34,041 - ml.utils.pipelines.database_handler - INFO - Connexion à la base de données vérifiée
2026-06-12 04:46:34,290 - ml.utils.pipelines.database_handler - INFO - Colonne actual_value ajoutée ou déjà existante
2026-06-12 04:46:34,292 - ml.utils.pipelines.Actual_values_pipeline - INFO - Base de données configurée


✅ Pipeline configuré avec succès
  - Connexion à la base de données vérifiée
  - Colonne actual_value ajoutée si nécessaire


## 4. Récupération des Prédictions de la Veille

In [5]:
from datetime import datetime, timedelta

# Calculer la date de la veille
today = datetime.now().date()
yesterday = today - timedelta(days=1)

print(f"📅 Récupération des prédictions du {yesterday}")
print(f"  Date actuelle: {today}")

try:
    if not pipeline.get_previous_day_predictions():
        print(f"⚠️ Aucune prédiction trouvée pour la date {yesterday}")
        print("   Assurez-vous que le pipeline de prédiction a été exécuté pour cette date.")
    else:
        print(f"✅ {len(pipeline.previous_day_predictions)} prédictions récupérées")
        
        # Afficher un aperçu des prédictions
        import pandas as pd
        display(pipeline.previous_day_predictions.head())
        
except Exception as e:
    logger.error(f"Erreur lors de la récupération des prédictions: {e}")
    raise

2026-06-12 04:46:34,321 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 2: RÉCUPÉRATION DES PRÉDICTIONS DE LA VEILLE ===
2026-06-12 04:46:34,324 - ml.utils.pipelines.Actual_values_pipeline - INFO - Récupération des prédictions du 2026-06-11
2026-06-12 04:46:34,326 - ml.utils.pipelines.Actual_values_pipeline - INFO - Plage de temps: 2026-06-11 00:00:00 à 2026-06-11 23:59:59.999999


📅 Récupération des prédictions du 2026-06-11
  Date actuelle: 2026-06-12


2026-06-12 04:46:34,635 - ml.utils.pipelines.Actual_values_pipeline - WARNING - Aucune prédiction trouvée pour la date 2026-06-11


⚠️ Aucune prédiction trouvée pour la date 2026-06-11
   Assurez-vous que le pipeline de prédiction a été exécuté pour cette date.


## 5. Génération de Valeurs Aléatoires

In [6]:
print("🎲 Génération de valeurs aléatoires...")
print("  (variation de ±20% autour de la valeur prédite)")

try:
    if pipeline.previous_day_predictions is None or len(pipeline.previous_day_predictions) == 0:
        print("⚠️ Aucune prédiction à traiter")
    else:
        if not pipeline.generate_random_actual_values():
            raise RuntimeError("Échec de la génération des valeurs aléatoires")
        
        print(f"✅ {pipeline.updated_count} valeurs aléatoires générées et mises à jour")
        
except Exception as e:
    logger.error(f"Erreur lors de la génération des valeurs: {e}")
    raise

🎲 Génération de valeurs aléatoires...
  (variation de ±20% autour de la valeur prédite)
⚠️ Aucune prédiction à traiter


## 6. Vérification des Mises à Jour

In [7]:
print("🔍 Vérification des mises à jour...")

try:
    updated_predictions = pipeline.verify_updates()
    
    if updated_predictions is None:
        print("⚠️ Impossible de vérifier les mises à jour")
    else:
        # Filtrer les enregistrements avec des valeurs réelles
        with_actual_values = updated_predictions[updated_predictions['actual_value'].notna()]
        
        print(f"✅ Vérification terminée")
        print(f"  Total prédictions: {len(updated_predictions)}")
        print(f"  Avec valeurs réelles: {len(with_actual_values)}")
        
        if len(with_actual_values) > 0:
            print("\n📊 Aperçu des prédictions mises à jour:")
            display(with_actual_values[['prediction_timestamp', 'prediction', 'actual_value']].head(10))
            
            # Calculer les statistiques
            print("\n📈 Statistiques de comparaison:")
            print(f"  Moyenne prédite: {with_actual_values['prediction'].mean():.2f}")
            print(f"  Moyenne réelle: {with_actual_values['actual_value'].mean():.2f}")
            print(f"  Écart moyen: {abs(with_actual_values['prediction'] - with_actual_values['actual_value']).mean():.2f}")
            print(f"  Écart relatif moyen: {(abs(with_actual_values['prediction'] - with_actual_values['actual_value']) / with_actual_values['prediction'] * 100).mean():.2f}%")
        
except Exception as e:
    logger.error(f"Erreur lors de la vérification: {e}")
    raise

2026-06-12 04:46:34,700 - ml.utils.pipelines.Actual_values_pipeline - INFO - === ÉTAPE 4: VÉRIFICATION DES MISES À JOUR ===


🔍 Vérification des mises à jour...


2026-06-12 04:46:34,926 - __main__ - ERROR - Erreur lors de la vérification: 'actual_value'


KeyError: 'actual_value'

## 7. Exécution Complète du Pipeline

In [ ]:
print("🚀 Exécution complète du pipeline de valeurs réelles...")

try:
    success, results = pipeline.run_full_pipeline()
    
    if success:
        print("\n✅ Pipeline exécuté avec succès")
        print(f"  Enregistrements mis à jour: {pipeline.updated_count}")
    else:
        print("\n❌ Pipeline échoué")
        
except Exception as e:
    logger.error(f"Erreur lors de l'exécution du pipeline: {e}")
    raise

## 8. Utilisation du Flow Prefect (Optionnel)

In [ ]:
# Optionnel : Utiliser le flow Prefect pour l'exécution
# from ml.workflows.actual_values_flow import actual_values_full_pipeline
# 
# result = actual_values_full_pipeline(db_uri=DB_URI)
# print(f"Résultat du flow: {result}")

print("💡 Pour utiliser le flow Prefect, décommentez le code ci-dessus")

## 9. Nettoyage et Statistiques Finales

In [ ]:
# Statistiques finales de la base de données
from ml.utils.pipelines.database_handler import DatabaseHandler

db_handler = DatabaseHandler(DB_URI)
stats = db_handler.get_prediction_stats()

if stats:
    print("📊 Statistiques finales de la base de données:")
    print(f"  Total des prédictions: {stats['total_predictions']}")
    print(f"  Table existe: {stats['table_exists']}")
else:
    print("⚠️ Impossible de récupérer les statistiques")

## Résumé

Ce notebook a démontré :
1. ✅ Configuration du pipeline de valeurs réelles
2. ✅ Récupération des prédictions de la veille
3. ✅ Génération de valeurs aléatoires (±20% de variation)
4. ✅ Mise à jour des enregistrements dans la base de données
5. ✅ Vérification des mises à jour

## Prochaines étapes
- Remplacer la génération aléatoire par l'intégration réelle avec les données de consommation
- Configurer le scheduling automatique via Prefect
- Ajouter des alertes en cas d'écart important entre prédiction et réalité